# Convex Hull Editor Demo

This notebook demonstrates a reproducible workflow for editing a DFT convex hull based on experimental information 
(e.g. calorimetric assessment) and propagating the assessed formation enthalpies into `gliquid` predictions:

1. Load a baseline system and inspect predicted behavior.
2. Edit the convex hull interactively.
3. Recompute feature values used by the production model.
4. Re-run predictions with the modified hull.

In [ ]:
import os
import gliquid.config as cfg
from gliquid.production_model_runner import ProductionModelRunner

# Needed only when fetching uncached MP data.
os.environ["NEW_MP_API_KEY"] = "YOUR_API_KEY_HERE"

# Bundle with trained production models + feature metadata.
runner = ProductionModelRunner.from_data_dir(cfg.data_dir)


In [ ]:
from pymatgen.analysis.phase_diagram import PhaseDiagram
from gliquid.load_binary_data import melt_enthalpies

EV_TO_JMOL = 96_485.0  # 1 eV/atom  →  J/mol

def compute_chull_energy_features(
    ch: PhaseDiagram,
    lhs: str,
    rhs: str,
    melt_enth: dict,
    vaporization_enth: dict | None = None,
) -> dict:
    """Compute the DFT convex-hull features that depend *only* on phase
    formation energies and elemental melting enthalpies.

    Symmetric features returned
    ---------------------------
    min_form_enthalpy_ch        - min formation enthalpy (J/mol) among stable
                                  phases (falls back to metastable if all ≥ 0)
    ave_metastable_melting_enth - mean (E_f - melt-line) for metastable compounds

    Antisymmetric features returned  (AB orientation, comp = x_rhs)
    ----------------------------------------------------------------
    skew_ch_enthalpy              - Σ (x_rhs - 0.5)·|H_f| over stable phases
    min_form_enth_ch_comp         - x_rhs of the minimum-enthalpy phase
    skew_metastable_melting_enth  - composition-weighted skew of metastable
                                   energy differences from the melt line

    Parameters
    ----------
    ch : PhaseDiagram
        pymatgen PhaseDiagram (original or modified).
    lhs, rhs : str
        Element symbols.  *rhs* is the solute whose fractional composition
        defines the "comp" axis, matching Generate_ML_Dataset.py conventions.
    melt_enth : dict
        {element_symbol: ΔH_fusion in J/mol}.
    vaporization_enth : dict, optional
        {element_symbol: ΔH_vap in J/mol}.  When provided, metastable phases
        above the vaporization-enthalpy line are excluded (same filter used
        during training-set generation).

    Returns
    -------
    dict with keys ``'symmetric'``, ``'antisymmetric_AB'``,
    ``'antisymmetric_BA'``.  Each value is a ``{feature_name: value}`` dict.
    """

    # ── stable phases (enthalpy + composition only) ──────────────────────
    stable_phases = []
    for entry in ch.stable_entries:
        comp_rhs = entry.composition.fractional_composition.as_dict().get(rhs, 0)
        enthalpy = EV_TO_JMOL * ch.get_form_energy_per_atom(entry)
        is_elemental = len(entry.composition.elements) == 1
        stable_phases.append(
            {"name": entry.name, "comp": comp_rhs,
             "enthalpy": enthalpy, "is_elemental": is_elemental}
        )

    # ── metastable compounds (≥ 2 elements, not on hull) ────────────────
    metastable_entries = [e for e in ch.all_entries if e not in ch.stable_entries]
    metastable_compounds = [e for e in metastable_entries
                            if len(e.composition.elements) > 1]

    metastable_metrics: list[dict] = []
    for entry in metastable_compounds:
        comp_dict = entry.composition.fractional_composition.as_dict()
        x_lhs = comp_dict.get(lhs, 0)
        x_rhs = comp_dict.get(rhs, 0)
        phase_energy = EV_TO_JMOL * ch.get_form_energy_per_atom(entry)

        # optional vaporization filter (mirrors Generate_ML_Dataset)
        if vaporization_enth is not None:
            vap_limit = (x_lhs * vaporization_enth[lhs]
                         + x_rhs * vaporization_enth[rhs])
            if phase_energy > vap_limit:
                continue

        melt_line = x_lhs * melt_enth[lhs] + x_rhs * melt_enth[rhs]
        metastable_metrics.append(
            {"comp": x_rhs,
             "energy_diff_from_melt_line": phase_energy - melt_line,
             "enthalpy": phase_energy}
        )
    metastable_metrics.sort(key=lambda m: m["comp"])

    # ── ave_metastable_melting_enth  (symmetric) ────────────────────────
    if metastable_metrics:
        avg_energy_below_melt = (
            sum(m["energy_diff_from_melt_line"] for m in metastable_metrics)
            / len(metastable_metrics)
        )
        total_skew = sum(
            (m["comp"] - 0.5) * m["energy_diff_from_melt_line"]
            for m in metastable_metrics
        )
        energy_comp_skew = total_skew / len(metastable_metrics)
    else:
        avg_energy_below_melt = 0.0
        energy_comp_skew = 0.0

    # ── skew_ch_enthalpy  (antisymmetric) ───────────────────────────────
    skew_ch_enthalpy = sum(
        (p["comp"] - 0.5) * abs(p["enthalpy"]) for p in stable_phases
    )

    # ── min_form_enthalpy_ch  (symmetric) & min_form_enth_ch_comp (anti)
    min_stable = min(stable_phases, key=lambda p: p["enthalpy"])
    min_stable_forme = min_stable["enthalpy"]
    min_stable_comp = (min_stable["comp"]
                       if not min_stable["is_elemental"] else 0.5)

    if metastable_metrics:
        min_ms = min(metastable_metrics, key=lambda m: m["enthalpy"])
        min_ms_forme = min_ms["enthalpy"]
        min_ms_comp  = min_ms["comp"]
    else:
        min_ms_forme = 0.0
        min_ms_comp  = 0.0

    min_form_enthalpy = (min_stable_forme
                         if min_stable_forme < 0 else min_ms_forme)

    if min_stable_comp not in (0, 1):
        min_form_comp = min_stable_comp
    elif min_ms_comp not in (0, 1):
        min_form_comp = min_ms_comp
    else:
        min_form_comp = 0.5

    # ── assemble output ─────────────────────────────────────────────────
    symm = {
        "min_form_enthalpy_ch":       min_form_enthalpy,
        "ave_metastable_melting_enth": avg_energy_below_melt,
    }
    anti_AB = {
        "skew_ch_enthalpy":             skew_ch_enthalpy,
        "min_form_enth_ch_comp":        min_form_comp,
        "skew_metastable_melting_enth": energy_comp_skew,
    }
    anti_BA = {
        "skew_ch_enthalpy":             -skew_ch_enthalpy,
        "min_form_enth_ch_comp":        1 - min_form_comp,
        "skew_metastable_melting_enth": -energy_comp_skew,
    }
    return {"symmetric": symm,
            "antisymmetric_AB": anti_AB,
            "antisymmetric_BA": anti_BA}


def update_runner_chull_features(
    runner,
    system: str,
    modified_pd: PhaseDiagram,
    melt_enth: dict | None = None,
    vaporization_enth: dict | None = None,
) -> dict:
    """Recompute convex-hull energy features from *modified_pd* and patch
    the runner's prediction DataFrames in-place.

    Parameters
    ----------
    runner : ProductionModelRunner
    system : str   - e.g. ``"Ga-Ru"`` (alphabetically sorted A-B form)
    modified_pd : PhaseDiagram
    melt_enth : dict, optional
        Defaults to ``gliquid.load_binary_data.melt_enthalpies``.
    vaporization_enth : dict, optional
        Passed through to ``compute_chull_energy_features``.

    Returns
    -------
    dict - the raw feature dict produced by ``compute_chull_energy_features``.
    """
    if melt_enth is None:
        melt_enth = melt_enthalpies

    lhs, rhs = system.split("-")
    mirror = f"{rhs}-{lhs}"

    feats = compute_chull_energy_features(
        modified_pd, lhs, rhs, melt_enth, vaporization_enth
    )

    # ── patch symmetric df ──────────────────────────────────────────────
    mask_s = runner.df_symm["system"] == system
    for col, val in feats["symmetric"].items():
        runner.df_symm.loc[mask_s, col] = val

    # ── patch antisymmetric df (A-B row) ────────────────────────────────
    mask_ab = runner.df_anti["system"] == system
    for col, val in feats["antisymmetric_AB"].items():
        runner.df_anti.loc[mask_ab, col] = val

    # ── patch antisymmetric df (B-A row) ────────────────────────────────
    mask_ba = runner.df_anti["system"] == mirror
    for col, val in feats["antisymmetric_BA"].items():
        runner.df_anti.loc[mask_ba, col] = val

    return feats

## 1 - Plot predicted liquidus curve and SHAP analysis

Use latest ML model ensemble to predict non-ideal mixing parameters for any binary system and plot the resulting phase diagram and SHAP figure, which displays the most impactful features for each predicted parameter and the effect of each feature on the prediction

In [ ]:
from gliquid.binary import BinaryLiquid, BLPlotter

system = "Ga-Ru"   # <-- change this to any cached binary system

pred_params = runner.predict_system(system) + [0]
print(pred_params)
bl = BinaryLiquid.from_cache(system, reconstruction=True)
bl.update_params(pred_params)
blp = BLPlotter(bl)

blp.show('ch+g')
blp.show('pred+liq')

runner.create_compact_prediction_figure(
    system_name=system,
    output_file=None,
    max_display_features=10,
)

pd = bl.dft_ch
print("Stable entries:")
for e in sorted(pd.stable_entries, key=lambda e: e.composition.get_atomic_fraction(pd.elements[1])):
    print(f"  {e.name:12s}  x={e.composition.get_atomic_fraction(pd.elements[1]):.3f}  "
          f"Ef={pd.get_form_energy_per_atom(e):+.4f} eV/at")

## 2 - Launch the editor

The widget includes:
- An interactive convex hull plot (hover for details)
- A summary table of entries with energies and hull distances
- Controls to modify, add, or remove entries
- Undo and reset buttons
- A change log

All formation energies are in **eV/atom**.

In [ ]:
from gliquid.hull_editor import ConvexHullEditor

editor = ConvexHullEditor(pd)
editor.display()

## 3 - Retrieve the modified phase diagram

After editing in the widget, run the next cell to get the updated `PhaseDiagram` object for downstream analysis.

In [ ]:
new_pd = editor.get_phase_diagram()
print(new_pd)
print("\nStable entries (modified):")
for e in sorted(new_pd.stable_entries, key=lambda e: e.composition.get_atomic_fraction(new_pd.elements[1])):
    print(f"  {e.name:12s}  x={e.composition.get_atomic_fraction(new_pd.elements[1]):.3f}  "
          f"Ef={new_pd.get_form_energy_per_atom(e):+.4f} eV/at")

In [ ]:
# Sanity check: compare computed features vs. the original dataframe values
lhs, rhs = system.split("-")
feats = compute_chull_energy_features(new_pd, lhs, rhs, melt_enthalpies)

symm_cols  = list(feats["symmetric"].keys())
anti_cols  = list(feats["antisymmetric_AB"].keys())

print(f"System: {system}\n")

# Symmetric features
row_s = runner.df_symm[runner.df_symm["system"] == system]
print("Symmetric features:")
for col in symm_cols:
    orig = row_s[col].values[0]
    new  = feats["symmetric"][col]
    diff = abs(orig - new)
    ok   = "✓" if diff < 1e-6 else f"✗  Δ={diff:.6f}"
    print(f"  {col:30s}  orig={orig:+14.4f}  recomp={new:+14.4f}  {ok}")

# Antisymmetric features (A-B)
row_ab = runner.df_anti[runner.df_anti["system"] == system]
print(f"\nAntisymmetric features ({system}):")
for col in anti_cols:
    orig = row_ab[col].values[0]
    new  = feats["antisymmetric_AB"][col]
    diff = abs(orig - new)
    ok   = "✓" if diff < 1e-6 else f"✗  Δ={diff:.6f}"
    print(f"  {col:30s}  orig={orig:+14.4f}  recomp={new:+14.4f}  {ok}")

# Antisymmetric features (B-A)
mirror = f"{rhs}-{lhs}"
row_ba = runner.df_anti[runner.df_anti["system"] == mirror]
print(f"\nAntisymmetric features ({mirror}):")
for col in anti_cols:
    orig = row_ba[col].values[0]
    new  = feats["antisymmetric_BA"][col]
    diff = abs(orig - new)
    ok   = "✓" if diff < 1e-6 else f"✗  Δ={diff:.6f}"
    print(f"  {col:30s}  orig={orig:+14.4f}  recomp={new:+14.4f}  {ok}")

## 4 - Optional: plug into BinaryLiquid

Replace the original DFT hull in a `BinaryLiquid` object, refresh model features, and regenerate predictions.

In [ ]:
from gliquid.binary import BinaryLiquid, build_phases_from_chull, BLPlotter

bl = BinaryLiquid.from_cache(system, param_format='comb-exp')

# Swap in the modified convex hull
bl.dft_ch = new_pd
bl.phases = build_phases_from_chull(new_pd, bl.components[1])
# Update runner's convex hull features with the modified phase diagram
update_runner_chull_features(runner, system, new_pd, melt_enthalpies)

# Re-run prediction with updated features
pred_params = runner.predict_system(system) + [0]
print("Updated predicted parameters:", pred_params)
bl.update_params(pred_params)

blp = BLPlotter(bl)
blp.show('ch+g')
blp.show('pred+liq')

runner.create_compact_prediction_figure(
    system_name=system,
    output_file=None,
    max_display_features=10,
)